# FB15k-237 — full pipeline: TransE/RotatE baselines + custom negative sampling

Self-contained Colab notebook that reproduces every result in our D3 / DL-final
submissions:
* Section D — **Baselines**: TransE and RotatE trained with PyKEEN's pipeline under the
  unified configuration (NSSALoss, Bernoulli filtered sampler, 8 negatives per positive).
* Section E — **Custom RotatE**: two-stage negative sampling (`random`, `hard`, `mixed`)
  drawn from a scored candidate pool, same loss / sampler / hyperparameters as the baselines.
* Section F — **Sliced evaluation** on every checkpoint (per relation frequency, head /
  tail entity degree, head vs tail prediction).
* Section G — **Qualitative examples**: top-K predictions on 5 sampled test triples.
* Section H — **Headline figures** for the report.

## Unified configuration (matches M1 feedback on matched hyperparameters)
All 7 runs share:
* Loss: `NSSALoss(margin=9.0, adversarial_temperature=1.0)` — RotatE paper default.
* Sampler: Bernoulli with `filtered=True`, `num_negs_per_pos=8`.
* Model: `embedding_dim=128`, Adam `lr=1e-3`, `batch_size=1024`.
* Training: 50 epochs max, early stopping on filtered val MRR, `patience=10`.
* Seed: 42.

Only the **negative-selection strategy** changes between runs.

## How to use
1. Set Colab runtime to **GPU (T4)** (Runtime ▸ Change runtime type).
2. Run all cells in order. Section A is a one-time install.
3. Section C runs a **2-minute smoke** of the custom pipeline.
4. Sections D and E hold the **full 50-epoch commands**, commented out by default
   (uncomment what you want to re-train; total ≈ 6 h on T4).
5. Sections F–H produce the post-hoc analyses on any available checkpoint.


## A. Setup (clone repo + install dependencies)

If the repo is private and the clone fails, the cell falls back to manual upload:
create `/content/dl_kg_project/code/` in the Colab Files panel and drop the contents of
`code/` plus `requirements.txt` there.

In [ ]:
import os, subprocess

REPO_URL  = "https://github.com/thaalia/dl_kg_project.git"  # adapt if the path is different
REPO_PATH = "/content/dl_kg_project"
CODE_DIR  = os.path.join(REPO_PATH, "code")

if not os.path.isdir(CODE_DIR):
    if not os.path.isdir(REPO_PATH):
        result = subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_PATH],
            capture_output=True, text=True,
        )
        if result.returncode != 0:
            os.makedirs(CODE_DIR, exist_ok=True)
            print("git clone failed (private repo or no network access).")
            print("Manual upload required:")
            print(f"  1. In the Colab Files panel, navigate to {REPO_PATH}/code/")
            print("  2. Upload every .py file from the local code/ folder.")
            print("  3. Also upload requirements.txt at the repo root.")
            print("\nClone stderr:\n", result.stderr.strip())

os.chdir(REPO_PATH)
print("\nWorking directory:", os.getcwd())
if os.path.isdir("code"):
    print("Code files:", sorted(os.listdir("code")))
else:
    print("code/ still missing — please upload the Python files before continuing.")

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import torch
print("PyTorch        :", torch.__version__)
print("CUDA available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU            :", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Full training will be impractically slow on CPU.")

## B. Configuration

These constants mirror the CLI flags of `train_baseline_kge.py` and
`train_rotate_custom.py`. They are interpolated into the `!python …` calls below.

In [ ]:
# Shared training hyperparameters
DIM                     = 128
BATCH_SIZE              = 1024
LR                      = 1e-3
EPOCHS                  = 50
PATIENCE                = 10
SEED                    = 42
MARGIN                  = 9.0
ADVERSARIAL_TEMPERATURE = 1.0
NUM_NEGS                = 8

# Custom-pipeline-only parameters
POOL_SIZE               = 64
SMOKE_STRATEGY          = "mixed"
SMOKE_HARD_FRACTION     = 0.5

print(
    f"loss=NSSALoss(margin={MARGIN}, adv_temp={ADVERSARIAL_TEMPERATURE}) | "
    f"sampler=bernoulli(filtered=True, num_negs={NUM_NEGS}) | "
    f"d={DIM} bs={BATCH_SIZE} lr={LR} epochs={EPOCHS} patience={PATIENCE} seed={SEED}"
)

## C. Smoke test on the custom pipeline (~2 min)

Validates that the full pipeline (candidate generation → scoring → selection → loss
back-prop → save) runs end-to-end without errors. Uses a dedicated output folder
`artifacts/custom/RotatE_smoke/` so it can never overwrite a reported run.

In [ ]:
!python code/train_rotate_custom.py \
    --strategy {SMOKE_STRATEGY} \
    --hard-fraction {SMOKE_HARD_FRACTION} \
    --num-negs {NUM_NEGS} \
    --pool-size {POOL_SIZE} \
    --epochs 1 \
    --batch-size 256 \
    --margin {MARGIN} \
    --adversarial-temperature {ADVERSARIAL_TEMPERATURE} \
    --seed {SEED} \
    --limit-batches 10 \
    --limit-val-eval 200 \
    --run-label RotatE_smoke

## D. Baseline runs (TransE + RotatE, ≈ 80 min on T4)

Each command calls PyKEEN's `pipeline()` with our unified configuration
(`NSSALoss + Bernoulli filtered + 8 negs`) and writes to
`artifacts/baseline/{TransE,RotatE}_summary.txt`, `_curves.png`, and
`pykeen_<Model>/trained_model.pkl`.

These cells are commented out because each run already exists in `artifacts/baseline/`.
Uncomment to retrain from scratch.

In [ ]:
# !python code/train_baseline_kge.py --model TransE \
#     --epochs {EPOCHS} --batch_size {BATCH_SIZE} --lr {LR} --dim {DIM} \
#     --patience {PATIENCE} --margin {MARGIN} \
#     --adversarial-temperature {ADVERSARIAL_TEMPERATURE} --num-negs {NUM_NEGS}

# !python code/train_baseline_kge.py --model RotatE \
#     --epochs {EPOCHS} --batch_size {BATCH_SIZE} --lr {LR} --dim {DIM} \
#     --patience {PATIENCE} --margin {MARGIN} \
#     --adversarial-temperature {ADVERSARIAL_TEMPERATURE} --num-negs {NUM_NEGS}

## E. Custom RotatE runs — 5 negative-sampling strategies (≈ 4–5 h on T4)

These are the exact commands behind our reported numbers. Each one writes a folder
under `artifacts/custom/RotatE_<label>/` containing `summary.txt`, `history.json`,
`curves.png`, and `trained_model.pkl`. Do **not** run `git pull` between two runs that
you want to compare directly.

In [ ]:
# Uncomment one (or more) of these to launch the full 50-epoch runs.

# !python code/train_rotate_custom.py --strategy random \
#     --num-negs 8 --epochs 50 --patience 10

# !python code/train_rotate_custom.py --strategy hard \
#     --num-negs 8 --epochs 50 --patience 10

# !python code/train_rotate_custom.py --strategy mixed --hard-fraction 0.3 \
#     --num-negs 8 --epochs 50 --patience 10

# !python code/train_rotate_custom.py --strategy mixed --hard-fraction 0.5 \
#     --num-negs 8 --epochs 50 --patience 10

# !python code/train_rotate_custom.py --strategy mixed --hard-fraction 0.7 \
#     --num-negs 8 --epochs 50 --patience 10

## F. Sliced evaluation on every available checkpoint

`evaluate_slices.py` finds every `trained_model.pkl` under `artifacts/` and writes a
`slices.json` next to it. Buckets (low / mid / high) are computed once from the training
split and cached in `artifacts/slice_buckets.json`, so every run is evaluated against the
same partition.

In [ ]:
!python code/evaluate_slices.py --all

In [ ]:
# Consolidate every per-run slices.json into a single DataFrame.
import json
from pathlib import Path
import pandas as pd

rows = []
for slice_path in sorted(Path("artifacts").rglob("slices.json")):
    if "baseline_old_" in slice_path.as_posix() or "RotatE_smoke" in slice_path.as_posix():
        continue
    data = json.load(slice_path.open())
    g = data["global"]
    row = {
        "run": str(slice_path.parent.relative_to("artifacts")),
        "mrr": g["mrr"], "h@1": g["hits_at_1"], "h@3": g["hits_at_3"], "h@10": g["hits_at_10"],
        "mrr_head": g["mrr_head"], "mrr_tail": g["mrr_tail"],
    }
    for axis in ("relation_frequency", "head_degree", "tail_degree"):
        for bucket in ("low", "mid", "high"):
            row[f"{axis}.{bucket}"] = data[axis][bucket].get("mrr")
    rows.append(row)

df = pd.DataFrame(rows).set_index("run").round(4)
df

## G. Qualitative comparison on 5 sampled test triples

`qualitative_examples.py` samples 5 test triples (seed=42) and, for every checkpoint,
prints the filtered rank of the gold entity plus the top-K predicted candidates on both
sides. The full report is saved to `artifacts/qualitative.md`.

In [ ]:
!python code/qualitative_examples.py --num-triples 5

In [ ]:
# Preview the first ~120 lines of the qualitative report inline.
from IPython.display import Markdown
report = Path("artifacts/qualitative.md").read_text().splitlines()
Markdown("\n".join(report[:120]))

## H. Headline figures for the report

Two figures: (1) global MRR per run, (2) MRR vs `tail in-degree` bucket — the slice that
benefits most from hard-negative sampling.

In [ ]:
import matplotlib.pyplot as plt

if df.empty:
    print("No checkpoints found; run sections C–E first.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    df["mrr"].sort_values().plot.barh(ax=axes[0], color="steelblue")
    axes[0].set_title("Filtered MRR per run")
    axes[0].set_xlabel("MRR")
    rotate_baseline = df.loc[df.index.str.contains("pykeen_RotatE"), "mrr"]
    if not rotate_baseline.empty:
        axes[0].axvline(rotate_baseline.mean(), color="red", linestyle="--",
                        linewidth=1, label="RotatE baseline")
        axes[0].legend()

    cols = ["tail_degree.low", "tail_degree.mid", "tail_degree.high"]
    df[cols].T.plot(ax=axes[1], marker="o")
    axes[1].set_title("MRR by tail in-degree bucket")
    axes[1].set_xticks(range(3))
    axes[1].set_xticklabels(["low", "mid", "high"])
    axes[1].set_ylabel("Filtered MRR")
    axes[1].legend(loc="upper left", fontsize=8)
    fig.tight_layout()
    fig.savefig("artifacts/d3_headline_figures.png", dpi=150)
    plt.show()

## I. Notes for graders

* Two-stage negative sampling lives in three small modules:
  * `code/negative_sampling.py` — Bernoulli corruption + train-triple filter.
  * `code/score_candidates.py` — batched scoring of candidate triples.
  * `code/select_candidates.py` — `random` / `hard` / `mixed` selection from the scored pool.
* The two training drivers are `code/train_baseline_kge.py` (PyKEEN pipeline) and
  `code/train_rotate_custom.py` (custom loop). The latter calls
  `model.post_parameter_update()` after every optimiser step to keep RotatE's relation
  embeddings on the complex unit circle, mirroring what PyKEEN's pipeline does internally.
  This step was essential for the NSSALoss to converge under our custom loop.
* `code/evaluate_slices.py` and `code/qualitative_examples.py` are pure post-hoc tools that
  consume `trained_model.pkl` files and do not retrain anything.
* The full set of artefacts (summaries, learning-curve PNGs, `slices.json`, `qualitative.md`)
  is committed under `artifacts/` for reference.
